# 🥉 Bronze Currency — API Ingestion

## Goal
Extract currency exchange rate data from the Frankfurter API and load only new records into the Bronze Delta Table.

## Flow
Frankfurter API → PySpark DataFrame → Watermark → Incremental Filter → Bronze Delta Table

## 1. Configuration

In [0]:
START_DATE = "2010-12-29"
END_DATE = "2014-01-28"

BASE = "EUR"
QUOTES = "USD,GBP,CHF"

API_URL = "https://api.frankfurter.dev/v2/rates"

## 2. Extract Data from API

In [0]:
import requests

def extract_currency_data():
    response = requests.get(
        API_URL,
        params={
            "base": BASE,
            "quotes": QUOTES,
            "from": START_DATE,
            "to": END_DATE
        },
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    return data

## 3. Create Spark DataFrame

In [0]:
def create_dataframe(spark, data):
    df = spark.createDataFrame(data)

    return df

## 4. Get Watermark

In [0]:
def get_watermark():
    try:
        query = spark.sql("""
            SELECT MAX(date) AS watermark
            FROM bronze.currency_rates
        """)

        watermark = query.collect()[0]["watermark"]

        return watermark

    except:
        return None

## 5. Filter New Data

In [0]:
from pyspark.sql.functions import col

def filter_new_data(df, watermark):
    if watermark is None:
        new_data = df
    else:
        new_data = df.filter(
            col("date") > watermark
        )

    return new_data

## 6. Write to Bronze Delta Table

In [0]:
def write_bronze(new_data):
    new_data.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("bronze.currency_rates")

## 7. Run Bronze Pipeline

In [0]:
data = extract_currency_data()

df = create_dataframe(spark, data)

watermark = get_watermark()

new_data = filter_new_data(df, watermark)

write_bronze(new_data)

In [0]:
%sql
SELECT *
FROM bronze.currency_rates;